# Hyperparameter Tuning for Irrigation Need Models
This notebook collects the tuning workflows from the existing notebooks for SGD, GBC, RandomForest, AdaBoost, Decision Tree, KNN, and XGBoost.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


In [ ]:
from pathlib import Path
import os

cwd = Path.cwd()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'data' / 'playground-series-s6e4' / 'train.csv').exists():
    repo_root = repo_root.parent

data_path = repo_root / 'data' / 'playground-series-s6e4' / 'train.csv'
if not data_path.exists():
    raise FileNotFoundError(f'Cannot find data/playground-series-s6e4/train.csv from {cwd} or any parent directory')

print('Using dataset:', data_path)


In [ ]:

param_dist = {
    'loss': ['hinge', 'log_loss', 'modified_huber'],
    'max_iter': [1000, 2000, 5000],
    'penalty': ['l2', 'l1', None],
    'alpha': [1e-4, 1e-3, 1e-2],
    'l1_ratio': [0.15, 0.5, 0.85],
    'learning_rate': ['optimal', 'constant', 'adaptive'],
    'eta0': [0.1, 1.0]
}

sgd = SGDClassifier(class_weight='balanced', tol=1e-3, random_state=50)
search_sgd = RandomizedSearchCV(estimator=sgd, param_distributions=param_dist, cv=5, n_jobs=-1, scoring='f1_macro', random_state=50)
search_sgd.fit(X_train.select_dtypes(include=['int64', 'float64']), y_train)
print('Best SGD parameters:', search_sgd.best_params_)
print('Best SGD score:', search_sgd.best_score_)


In [ ]:

from sklearn.utils.class_weight import compute_sample_weight
gbc = GradientBoostingClassifier(random_state=50)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
param_dist = {
    'n_estimators': [50, 100, 300],
    'learning_rate': [0.05, 0.2],
    'max_depth': [3, 5],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [2, 4],
    'subsample': [0.6, 1.0],
    'max_features': ['sqrt', 'log2']
}
search_gbc = RandomizedSearchCV(estimator=gbc, param_distributions=param_dist, n_iter=30, cv=5, scoring='f1_macro', n_jobs=-1, random_state=50)
search_gbc.fit(X_train.select_dtypes(include=['int64', 'float64']), y_train, sample_weight=sample_weights)
print('Best GBC parameters:', search_gbc.best_params_)
print('Best GBC score:', search_gbc.best_score_)


In [ ]:

rf = RandomForestClassifier()
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [30, 35, 40],
    'max_features': ['sqrt', 'log2', 0.6],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True]
}
search_rf = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=30, cv=3, scoring='f1_macro', n_jobs=-1, random_state=50)
search_rf.fit(X_train.select_dtypes(include=['int64', 'float64']), y_train)
print('Best RF params:', search_rf.best_params_)
print('Best RF score:', search_rf.best_score_)


In [ ]:

ada = AdaBoostClassifier()
param_dist = {
    'n_estimators': [50, 100, 120],
    'learning_rate': [0.1, 0.5, 1.0]
}
search_ada = RandomizedSearchCV(ada, param_distributions=param_dist, cv=3, scoring='accuracy', n_jobs=-1, random_state=50)
search_ada.fit(X_train.select_dtypes(include=['int64', 'float64']), y_train)
print('Best AdaBoost params:', search_ada.best_params_)
print('Best AdaBoost score:', search_ada.best_score_)


In [ ]:

param_dist = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features': [None, 'sqrt', 'log2']
}
dt = DecisionTreeClassifier(class_weight='balanced', random_state=50)
search_dt = RandomizedSearchCV(dt, param_distributions=param_dist, n_iter=20, scoring='f1_macro', cv=5, n_jobs=-1, random_state=50)
search_dt.fit(X_train.select_dtypes(include=['int64', 'float64']), y_train)
print('Best DT params:', search_dt.best_params_)
print('Best DT score:', search_dt.best_score_)


In [ ]:

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
preprocessor_knn = ColumnTransformer(transformers=[('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
knn_pipeline = ImbPipeline([('preprocessing', preprocessor_knn), ('smote', SMOTE(random_state=50)), ('model', KNeighborsClassifier())])
param_dist_knn = {
    'model__n_neighbors': [3, 5, 7, 9, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}
search_knn = RandomizedSearchCV(knn_pipeline, param_distributions=param_dist_knn, n_iter=10, scoring='f1_weighted', cv=3, random_state=50, n_jobs=-1)
search_knn.fit(X_train, y_train)
print('Best KNN params:', search_knn.best_params_)
print('Best KNN score:', search_knn.best_score_)


In [ ]:

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
preprocessor_xgb = ColumnTransformer(transformers=[('num', 'passthrough', num_cols), ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)])
xgb_pipeline = Pipeline([('preprocessing', preprocessor_xgb), ('model', XGBClassifier(objective='multi:softmax', num_class=3, eval_metric='mlogloss', random_state=50, n_jobs=-1))])
param_dist_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}
search_xgb = RandomizedSearchCV(estimator=xgb_pipeline, param_distributions=param_dist_xgb, n_iter=10, scoring='f1_macro', cv=3, random_state=50, n_jobs=-1, verbose=2)
search_xgb.fit(X_train, y_train)
print('Best XGB params:', search_xgb.best_params_)
print('Best XGB score:', search_xgb.best_score_)
